# **Import**


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from data_loader import Dataset
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, mean_squared_error

# **Build From Scratch**


In [2]:
def _generate_sample(X, y, max_features):

    num_samples, num_features = X.shape

    if num_features > max_features:
        tree_feature_idxs = np.random.choice(num_features, max_features, replace=False)
    else:
        tree_feature_idxs = np.arange(num_features)

    X = X[:, tree_feature_idxs]

    idxs = np.random.choice(num_samples, num_samples, replace=True)

    return X[idxs], y[idxs], tree_feature_idxs

In [3]:
class CustomRandomForest():

    def __init__(self, num_trees=10, max_features=5, min_samples_split=5, max_depth=12):

        self.forest = []
        self.feature_idxs = []

        self.num_trees = num_trees
        self.max_features = max_features

        self.min_samples_split = min_samples_split
        self.max_depth = max_depth

    def fit(self, X, y):

        if len(set(y)) <= 20:
            self.problem_type = "classification"
        else:
            self.problem_type = "regression"

        for _ in range(self.num_trees):

            if self.problem_type == "classification":

                tree = DecisionTreeClassifier(
                    min_samples_split=self.min_samples_split,
                    max_depth=self.max_depth,
                    max_features=self.max_features
                )

            else:

                tree = DecisionTreeRegressor(
                min_samples_split=self.min_samples_split,
                max_depth=self.max_depth,
                max_features=self.max_features
            )

            X_sample, y_sample, tree_feature_idxs = _generate_sample(X, y, self.max_features)

            tree.fit(X_sample, y_sample)

            self.forest.append(tree)
            self.feature_idxs.append(tree_feature_idxs)

    def predict(self, X):

        tree_preds = []

        for tree, tree_feature_idxs in zip(self.forest, self.feature_idxs):

            X_subset = X[:, tree_feature_idxs]

            tree_pred = tree.predict(X_subset)

            tree_preds.append(tree_pred)

        tree_preds = np.array(tree_preds)

        predictions = [self._predict(tree_pred) for tree_pred in tree_preds.T]

        return np.array(predictions)

    def _predict(self, y):

        if self.problem_type == "classification":
            y = list(y)
            return max(y, key=y.count)
        else:
            return np.mean(y)

# **Load and Split**


In [4]:
dataset_m = Dataset("multiclass classification")
X_train_m, X_test_m, y_train_m, y_test_m = dataset_m.load_split_data()

dataset_b = Dataset("binary classification")
X_train_b, X_test_b, y_train_b, y_test_b = dataset_b.load_split_data()

dataset_r = Dataset("regression")
X_train_r, X_test_r, y_train_r, y_test_r = dataset_r.load_split_data()

# **Train, Test and Compare**



In [5]:
custom_model_m = CustomRandomForest()
custom_model_m.fit(X_train_m, y_train_m)
y_pred_custom_m = custom_model_m.predict(X_test_m)
print(f"Custom Multiclass Classification Accuracy: {accuracy_score(y_test_m, y_pred_custom_m):.3f}")

custom_model_b = CustomRandomForest()
custom_model_b.fit(X_train_b, y_train_b)
y_pred_custom_b = custom_model_b.predict(X_test_b)
print(f"Custom Binary Classification Accuracy: {accuracy_score(y_test_b, y_pred_custom_b):.3f}")

custom_model_r = CustomRandomForest()
custom_model_r.fit(X_train_r, y_train_r)
y_pred_custom_r = custom_model_r.predict(X_test_r)
print(f"Custom Regression MSE: {mean_squared_error(y_test_r, y_pred_custom_r):.3f}")

Custom Multiclass Classification Accuracy: 0.967
Custom Binary Classification Accuracy: 0.985
Custom Regression MSE: 0.306


In [6]:
sklearn_model_m = RandomForestClassifier()
sklearn_model_m.fit(X_train_m, y_train_m)
y_pred_sklearn_m = sklearn_model_m.predict(X_test_m)
print(f"Scikit-learn Multiclass Classification Accuracy: {accuracy_score(y_test_m, y_pred_sklearn_m):.3f}")

sklearn_model_b = RandomForestClassifier()
sklearn_model_b.fit(X_train_b, y_train_b)
y_pred_sklearn_b = sklearn_model_b.predict(X_test_b)
print(f"Scikit-learn Binary Classification Accuracy: {accuracy_score(y_test_b, y_pred_sklearn_b):.3f}")

sklearn_model_r = RandomForestRegressor()
sklearn_model_r.fit(X_train_r, y_train_r)
y_pred_sklearn_r = sklearn_model_r.predict(X_test_r)
print(f"Scikit-learn Regression MSE: {mean_squared_error(y_test_r, y_pred_sklearn_r):.3f}")

Scikit-learn Multiclass Classification Accuracy: 0.967
Scikit-learn Binary Classification Accuracy: 0.993
Scikit-learn Regression MSE: 0.239
